# Lesson 1 — Talking to your model through LangChain

**Goal:** send one message to your local llama.cpp server and get the reply,
using LangChain's model object instead of raw `httpx`.

**Before running:** start your llama.cpp server so it's live at
`http://127.0.0.1:8080`, and make sure this notebook's kernel is the
project `.venv` (top-right kernel picker in VS Code).

## The one concept: the *chat model* object

In your proxy you build a URL, build a JSON body, call `httpx`, and dig the
text back out of the JSON. LangChain wraps all of that in a single object
called a *chat model*. You hand it messages, it hands you back a message.

We use `ChatOpenAI` **not** because we're calling OpenAI, but because your
llama.cpp server speaks the OpenAI API dialect — the same
`/v1/chat/completions` shape your proxy forwards to. Any OpenAI-compatible
client can talk to it; we just point it at your local URL.

### The proxy parallel — `api/proxy/main.py:310-321`
```python
async with httpx.AsyncClient(timeout=timeout) as client:
    upstream = await client.post(url, json=payload.model_dump(exclude_none=True))
content = upstream.json()
assistant_text = content.get('choices', [{}])[0].get('message', {}).get('content', '')
```
That POST-and-dig is exactly what the cell below replaces.

In [ ]:
from langchain_openai import ChatOpenAI

# Each argument maps to something your proxy already deals with:
model = ChatOpenAI(
    base_url='http://127.0.0.1:8080/v1',  # your LLAMA_BASE_URL + /v1
    api_key='not-needed',                 # llama.cpp ignores it, but the client needs *some* value
    model='qwen3-14b-q4_k_m',             # your ALLOWED_MODEL from .env
    temperature=0.7,                      # same knob you'd pass in the JSON body
)
model

## Ask it something

`invoke` means "run this once, give me the result." We pass a list of
messages, each a `(role, content)` tuple — the same role/content pairs your
`Message` schema uses.

In [ ]:
response = model.invoke([
    ('system', 'You are a terse assistant. Answer in one sentence.'),
    ('human', 'What is a FastAPI dependency, in plain words?'),
])

# response is an AIMessage; the text lives in .content
print(response.content)

Everything else the model returned (token counts, finish reason) is in
`response_metadata`. Run the next cell to peek.

In [ ]:
response.response_metadata

---
**Try it:** change the `human` message and re-run the invoke cell. Notice you
never touched a URL, a JSON body, or `choices[0]` — that's the point.